In [1]:
import torch                    # Tensor 데이터의 타입으로 파싱하기 위해 로드
import torch.nn as nn           # torch 안에 nn부분만 로드하여 nn별칭으로 사용 (nn -> 기본뼈대)
import torch.optim as optim     # 옵티마이저 (기울기의 변화를 주는 기능)

In [2]:
# 데이터셋을 하나 생성
# 독립, 종속 데이터를 tensor로 생성
# 독립 변수 -> sklearn을 이용한 ML에서는 2차원 데이터
x = torch.tensor( [ [1.0], [2.0], [3.0], [4.0] ] )
# 종속 변수 -> sklearn을 이용한 ML에서는 1차원 -> torch에서는 2차원으로 생성
# 종속 변수는 독립 변수에서 2를 곱하고 1을 더한 값
y = torch.tensor( [ [3.0], [5.0], [7.0], [9.0] ] )

In [4]:
type(x)

torch.Tensor

In [5]:
# sklearn에서 모델을 생성한다면 -> 가중치가 2이고 절편이 1인 규칙을 찾는 LinearRegression을 이용하여 예측 가능

# torch을 이용한 단순 선형 회귀
# 순전파 ( 모델 학습 -> 예측 )
#       class 클래스명 ( 부모클래스 ):  --> 부모클래스의 기능을 상속 받아서 클래스를 선언
class LinearReg(nn.Module):
 
    # torch의 모듈을 이용한 클래스 생성시 2개의 함수를 필요로 선언 ( 생성자 함수, forward 함수 )

    # 생성자 함수
    def __init__(self):
        # self : 자기 자신( 클래스를 생성할 때 저장이 되는 위치 )
        # super() : 부모 클래스(nn.Module)를 의미
        super(LinearReg, self).__init__()       # 부모 클래스의 생성자 함수를 실행

        # 선형 회귀 모델을 이용
        # nn 모듈 안에 Linear() 모델은
        # 첫번째 인자값 : 입력 데이터 (독립 변수)의 차원(피쳐)의 수
        # 두번재 인자값 : 출력 데이터의 차원(피쳐)의 수
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

In [6]:
# 클래스 생성 -> 회귀 모델을 생성
model = LinearReg()

In [7]:
# 손실 함수
criterion = nn.MSELoss()

In [8]:
# 옵티마이저 설정 -> 가중치를 업데이트
# 어떤 모델의 파라미터를 설정할 것인가? -> 첫번째 인자
# lr 매개변수 -> 경사 하강법의 보폭
optimizer = optim.SGD(model.parameters(), lr = 0.01)

In [17]:
# 순전파 (생성된 모델을 호출하면 -> forward() 함수를 호출하도록 nn.Module에서 설정이 되어있음)
pred = model(x)
# LinearReg 클래스 안에 forward 함수를 호출하여 독립변수 (x)를 인자값으로 사용한다.

# 손실함수
loss = criterion(pred, y)

# 기울기를 초기화
optimizer.zero_grad()

# 역전파 (자동 미분) -> 데이터가 있는 쪽으로 방향을 제시한다. -> 네비게이션
loss.backward()

# 가중치를 업데이트 ( 파라미터(모델) 수정 )
optimizer.step()

# loss 값을 확인
print(loss)

tensor(18.3912, grad_fn=<MseLossBackward0>)


In [18]:
# DL 모델은 반복 학습이 기본 설정 -> 학습모드를 평가모드 전환
# eval() : 모델을 평가모드로 전환
# train() : 모델을 학습모드로 전환
model.eval()

# 예측, 평가 (메모리의 사용량을 줄이기 위해서 가중치의 계산을 잠시 비활성화)
with torch.no_grad():
    y_pred = model(x)
    loss = criterion(y_pred, y)
    print(y_pred)
    print(loss)

tensor([[1.0934],
        [2.0997],
        [3.1060],
        [4.1124]])
tensor(12.7747)


In [19]:
# 반복 학습을 통해서 가중치와 편향을 변화 시킨다.
epochs = 200
model.train()
for epoch in range(epochs):
    # 순전파
    pred = model(x)
    # 손실함수
    loss = criterion(pred, y)
    # 기울기 초기화
    optimizer.zero_grad()
    # 자동 미분(역전파) -> 가중치의 방향을 제시
    loss.backward()
    # 가중치를 업데이트
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        # 반복회수가 20회마다 출력
        print(f"Epoch : [ {epoch+1}, 200 ], Loss : {round(loss.item(), 6)}")

Epoch : [ 20, 200 ], Loss : 0.051852
Epoch : [ 40, 200 ], Loss : 0.035107
Epoch : [ 60, 200 ], Loss : 0.031132
Epoch : [ 80, 200 ], Loss : 0.027613
Epoch : [ 100, 200 ], Loss : 0.024493
Epoch : [ 120, 200 ], Loss : 0.021724
Epoch : [ 140, 200 ], Loss : 0.019269
Epoch : [ 160, 200 ], Loss : 0.017091
Epoch : [ 180, 200 ], Loss : 0.01516
Epoch : [ 200, 200 ], Loss : 0.013446


In [20]:
model.eval()

# 예측, 평가 (메모리의 사용량을 줄이기 위해서 가중치의 계산을 잠시 비활성화)
with torch.no_grad():
    y_pred = model(x)
    loss = criterion(y_pred, y)
    print(y_pred)
    print(loss)

tensor([[2.8133],
        [4.9095],
        [7.0058],
        [9.1020]])
tensor(0.0134)


In [21]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
# data = fetch_california_housing()

# x = data['data']
# y = data['target']

# print(x.shape, y.shape)

In [23]:
import pandas as pd

In [24]:
df = pd.read_csv('../csv/california_.csv')

x = df.drop('target', axis = 1).values
y = df['target'].values

In [26]:
# 1차원 데이터를 2차원으로 변경
y = y.reshape(-1, 1)

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [35]:
from sklearn.preprocessing import StandardScaler

In [28]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [39]:
scaler = StandardScaler()
X_train_sc = torch.tensor(scaler.fit_transform(X_train_tensor), dtype=torch.float32)
X_test_sc = torch.tensor(scaler.transform(X_test_tensor), dtype=torch.float32)

In [30]:
y_train_tensor

tensor([[1.0300],
        [3.8210],
        [1.7260],
        ...,
        [2.2210],
        [2.8350],
        [3.2500]])

In [31]:
# 선형 회귀 모델 객체를 선언
class Reg(nn.Module):
    # class 생성할때 입력 데이터의 피쳐의 수를 필수 인자로 설정
    def __init__(self, _dim):
        super(Reg, self).__init__()
        self.linear = nn.Linear(_dim, 1)
    
    def forward(self, x):
        return self.linear(x)

In [48]:
# 모델을 생성 -> 생성시 입력 데이터의 피쳐의 개수를 넣기
n_feature = X_train.shape[1]
model = Reg(n_feature)

In [49]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01)

In [50]:
# 반복 실행하면서 모델에 학습하고 가중치 업데이트
epochs = 300
for epoch in range(epochs):
    pred = model(X_train_sc)
    loss = criterion(pred, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    n = epoch + 1
    # 30회 마다 loss 확인
    if n % 30 == 0:
        print(f"Epoch : [{n} / 500], Loss : {round(loss.item(), 6)}")

Epoch : [30 / 500], Loss : 2.686963
Epoch : [60 / 500], Loss : 1.243695
Epoch : [90 / 500], Loss : 0.80935
Epoch : [120 / 500], Loss : 0.673294
Epoch : [150 / 500], Loss : 0.626328
Epoch : [180 / 500], Loss : 0.606466
Epoch : [210 / 500], Loss : 0.595242
Epoch : [240 / 500], Loss : 0.587103
Epoch : [270 / 500], Loss : 0.580343
Epoch : [300 / 500], Loss : 0.574405


In [51]:
# 학습된 모델을 이용하여 평가 검증

model.eval()

with torch.no_grad():
    pred = model(X_test_sc)
    loss = criterion(pred, y_test_tensor)

    print(round(loss.item(), 6))

0.58574


In [52]:
for i in range(10):
    print(f"실제 데이터 : {y_test[i]}, 예측 데이터 : {pred[i].item()}")

실제 데이터 : [0.477], 예측 데이터 : 0.9136018753051758
실제 데이터 : [0.458], 예측 데이터 : 1.6313228607177734
실제 데이터 : [5.00001], 예측 데이터 : 2.3299851417541504
실제 데이터 : [2.186], 예측 데이터 : 2.7668933868408203
실제 데이터 : [2.78], 예측 데이터 : 2.3717353343963623
실제 데이터 : [1.587], 예측 데이터 : 2.095721483230591
실제 데이터 : [1.982], 예측 데이터 : 2.7130250930786133
실제 데이터 : [1.575], 예측 데이터 : 2.176546812057495
실제 데이터 : [3.4], 예측 데이터 : 2.1513595581054688
실제 데이터 : [4.466], 예측 데이터 : 4.033158779144287


In [53]:
model2 = Reg(n_feature)
criterion2 = nn.MSELoss()
optimizer2 = optim.SGD(model2.parameters(), lr = 1e-07)

In [62]:
for epoch in range(300):
    pred2 = model2(X_train_tensor)
    loss2 = criterion2(pred2, y_train_tensor)
    optimizer2.zero_grad()
    loss2.backward()
    optimizer2.step()
    n = epoch + 1
    # 30회 마다 loss 확인
    if n % 30 == 0:
        print(f"Epoch : [{n} / 500], Loss : {round(loss2.item(), 6)}")

Epoch : [30 / 500], Loss : 20.363508
Epoch : [60 / 500], Loss : 20.067242
Epoch : [90 / 500], Loss : 19.791502
Epoch : [120 / 500], Loss : 19.534681
Epoch : [150 / 500], Loss : 19.295414
Epoch : [180 / 500], Loss : 19.07238
Epoch : [210 / 500], Loss : 18.864294
Epoch : [240 / 500], Loss : 18.670092
Epoch : [270 / 500], Loss : 18.488731
Epoch : [300 / 500], Loss : 18.319252


In [56]:
# 비선형 모델 생성 (선형 모델 -> 활성화 함수 -> 선형 모델)
class Reg2(nn.Module):
    def __init__(self, _dim):
        super(Reg2, self).__init__()
        # 다중 퍼셉트론 안에 선형 모델 -> 활성화 함수 -> 선형 모델
        self.model = nn.Sequential(
            # 첫번째 레이어
            nn.Linear(_dim, _dim),
            # 활성화 함수 (비선형 구조 파악) (ReLU(일반적으로 사용), Tanh, Sigmoid)
            nn.ReLU(),
            nn.Linear(_dim, 1)
        )
    def forward (self, x):
        return self.model(x)

In [60]:
model3 = Reg2(n_feature)
criterion3 = nn.MSELoss()
optimizer3 = optim.SGD(model3.parameters(), lr = 0.01)

In [65]:
# 반복 학습
for epoch in range(300):
    n = epoch + 1
    pred3 = model3(X_train_sc)
    loss3 = criterion3(pred3, y_train_tensor)
    optimizer3.zero_grad()
    loss3.backward()
    optimizer3.step()

    if n % 30 == 0:
        print(f"Epoch[{n} / 300, Loss : {round(loss3.item(), 6)}]")

Epoch[30 / 300, Loss : 0.504762]
Epoch[60 / 300, Loss : 0.502137]
Epoch[90 / 300, Loss : 0.499661]
Epoch[120 / 300, Loss : 0.497311]
Epoch[150 / 300, Loss : 0.495103]
Epoch[180 / 300, Loss : 0.493]
Epoch[210 / 300, Loss : 0.491]
Epoch[240 / 300, Loss : 0.489096]
Epoch[270 / 300, Loss : 0.487284]
Epoch[300 / 300, Loss : 0.48556]


In [66]:
model3.eval()
with torch.no_grad():       # 버릇처럼 사용 (메모리 최적화)
    pred3 = model3(X_test_sc)
    loss3 = criterion3(pred3, y_test_tensor)

for i in range(10):
    print(f"실제 데이터 : {y_test[i]}, 예측 데이터 : {pred[i].item()}")

실제 데이터 : [0.477], 예측 데이터 : 0.9136018753051758
실제 데이터 : [0.458], 예측 데이터 : 1.6313228607177734
실제 데이터 : [5.00001], 예측 데이터 : 2.3299851417541504
실제 데이터 : [2.186], 예측 데이터 : 2.7668933868408203
실제 데이터 : [2.78], 예측 데이터 : 2.3717353343963623
실제 데이터 : [1.587], 예측 데이터 : 2.095721483230591
실제 데이터 : [1.982], 예측 데이터 : 2.7130250930786133
실제 데이터 : [1.575], 예측 데이터 : 2.176546812057495
실제 데이터 : [3.4], 예측 데이터 : 2.1513595581054688
실제 데이터 : [4.466], 예측 데이터 : 4.033158779144287


In [67]:
# 파이토치 랜덤 고정
torch.manual_seed(42)

In [68]:
# 딥러닝 분류 모델
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

In [69]:
df = pd.read_csv("../csv/iris.csv")
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [70]:
from sklearn.preprocessing import LabelEncoder

In [71]:
# 선형 모델을 이용하여 분류 -> 로지스틱회귀랑 비슷한 방식
# 출력의 값이 3개의 피쳐로 출력 ()
x = df.drop('species', axis = 1)
y = df['species']

In [72]:
# y의 값들을 LabelEncoder를 이용하여 숫자로 변환
le = LabelEncoder()
y = le.fit_transform(y)
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [73]:
# train, test 데이터셋으로 분할
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [74]:
# Scaler 작업
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [75]:
# Tensor 형태로 변환
X_train_tensor = torch.tensor(X_train_sc, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_sc, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [76]:
# 모델 정의
class clf(nn.Module):
    def __init__(self, _dim):
        super(clf, self).__init__()
        self.model = nn.Linear(_dim, 3)

    def forward(self, x):
        return self.model(x)

In [77]:
clf_model = clf(x.shape[1])

In [79]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(clf_model.parameters(), lr = 0.01)

In [ ]:
pred = clf_model(X_train_tensor)
pred

In [ ]:
torch.max(pred, 1)

In [83]:
# 반복 학습
for epoch in range(300):
    pred = clf_model(X_train_tensor)
    loss = criterion(pred, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    n = epoch + 1
    if n % 30 == 0:
        print(f"Epoch : [{n} / 300], Loss : {round(loss.item(), 6)}")

Epoch : [30 / 300], Loss : 0.619521
Epoch : [60 / 300], Loss : 0.581321
Epoch : [90 / 300], Loss : 0.551174
Epoch : [120 / 300], Loss : 0.526589
Epoch : [150 / 300], Loss : 0.506034
Epoch : [180 / 300], Loss : 0.488507
Epoch : [210 / 300], Loss : 0.473322
Epoch : [240 / 300], Loss : 0.45999
Epoch : [270 / 300], Loss : 0.448155
Epoch : [300 / 300], Loss : 0.437545


In [84]:
# 평가
clf_model.eval()

with torch.no_grad():
    pred = clf_model(X_test_tensor)
    _, pred_idx = torch.max(pred, 1)

acc = accuracy_score(y_test, pred_idx)
print("정확도 : ", round(acc, 4))
print(classification_report(y_test, pred_idx))

정확도 :  0.8
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.40      0.57        10
           2       0.62      1.00      0.77        10

    accuracy                           0.80        30
   macro avg       0.88      0.80      0.78        30
weighted avg       0.88      0.80      0.78        30

